In [1]:
from pathlib import Path
import json
import polars as pl

In [3]:
PROJECT_ROOT = Path.cwd()

# If your notebook is inside /notebooks, move one level up to project root
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_GUILD_DIR = PROJECT_ROOT / "data" / "raw" / "tibiadata" / "guild"

snapshot_files = sorted(RAW_GUILD_DIR.rglob("*.json"))

print(f"Found {len(snapshot_files)} snapshot file(s).")

latest_snapshot = snapshot_files[-1]
print(f"Latest snapshot: {latest_snapshot}")

Found 1 snapshot file(s).
Latest snapshot: /Users/adrian/Documents/GitHub/tibia-guild-analytics/data/raw/tibiadata/guild/2026/05/14/black_clover_20260514T000004Z.json


In [4]:
with latest_snapshot.open("r", encoding="utf-8") as f:
    payload = json.load(f)

metadata = payload["metadata"]
data = payload["data"]

metadata

{'source': 'tibiadata',
 'entity_type': 'guild',
 'entity_name': 'Black Clover',
 'extracted_at_utc': '2026-05-14T00:00:04.036284+00:00'}

In [5]:
data.keys()

dict_keys(['guild', 'information'])

In [6]:
guild = data.get("guild", {})
guild.keys()

dict_keys(['name', 'world', 'logo_url', 'description', 'guildhalls', 'active', 'founded', 'open_applications', 'homepage', 'in_war', 'disband_date', 'disband_condition', 'players_online', 'players_offline', 'members_total', 'members_invited', 'members', 'invites'])

In [8]:
guild_summary = pl.DataFrame(
    [
        {
            "guild_name": guild.get("name"),
            "world": guild.get("world"),
            "description": guild.get("description"),
            "homepage": guild.get("homepage"),
            "founded": guild.get("founded"),
            "active": guild.get("active"),
            "guildhall": guild.get("guildhall"),
            "applications": guild.get("applications"),
            "member_count": len(guild.get("members", [])),
            "extracted_at_utc": metadata.get("extracted_at_utc"),
        }
    ]
)

guild_summary


guild_name,world,description,homepage,founded,active,guildhall,applications,member_count,extracted_at_utc
str,str,str,str,str,bool,null,null,i64,str
"""Black Clover""","""Lobera""","""hola somos un gremio amistoso …","""?action=externallinkwarning&am…","""2025-09-07""",true,null,null,96,"""2026-05-14T00:00:04.036284+00:…"


In [15]:
members = guild.get("members", [])

members_df = pl.DataFrame(members)

members_df.sort("level",  descending= True).head(10)

name,title,rank,vocation,level,joined,status
str,str,str,str,i64,str,str
"""Maxx Stark""","""""","""Black Soldier""","""Master Sorcerer""",698,"""2026-04-30""","""offline"""
"""Jefe Anthony""","""""","""Black Leader""","""Elite Knight""",697,"""2026-03-29""","""offline"""
"""Negin du Murin""","""""","""Black General""","""Elite Knight""",538,"""2025-10-04""","""offline"""
"""Duendemen Shelby""","""""","""Black Viceleader""","""Royal Paladin""",526,"""2026-04-18""","""online"""
"""Ek Bruta""","""""","""Black General""","""Elite Knight""",522,"""2025-11-05""","""offline"""
"""Trogodita""","""""","""Black Sages""","""Elder Druid""",520,"""2025-10-18""","""offline"""
"""Aleiz""","""""","""Black Sages""","""Royal Paladin""",513,"""2025-11-27""","""offline"""
"""Mong Not Rush""","""""","""Black General""","""Master Sorcerer""",478,"""2026-04-30""","""online"""
"""Kryvatu""","""""","""Black Sages""","""Elite Knight""",473,"""2026-02-13""","""offline"""


In [17]:
members = guild.get("members", [])

members_df = pl.DataFrame(members)

members_df.sort("level",  descending= False).head(10)

name,title,rank,vocation,level,joined,status
str,str,str,str,i64,str,str
"""El Noa Noa""","""""","""Black""","""Royal Paladin""",28,"""2026-04-02""","""offline"""
"""Brujita del barrio""","""""","""Black Soldier""","""Monk""",29,"""2025-12-31""","""offline"""
"""Masterelar""","""""","""Black Soldier""","""Knight""",33,"""2026-04-30""","""offline"""
"""Sirius Juan""","""""","""Black Soldier""","""Master Sorcerer""",41,"""2026-02-26""","""offline"""
"""Ulility""","""""","""Black""","""Royal Paladin""",45,"""2026-01-07""","""offline"""
"""Arepapower""","""""","""Black Sergeant""","""Exalted Monk""",55,"""2026-03-17""","""offline"""
"""Akonvolder""","""""","""Black Soldier""","""Master Sorcerer""",65,"""2026-04-30""","""offline"""
"""Legolas ligerus""","""""","""Black""","""Royal Paladin""",65,"""2025-12-10""","""offline"""
"""Azriana Klon""","""""","""Black""","""Royal Paladin""",70,"""2026-04-18""","""offline"""
